# AI工学101 — 第18回

## 交差検証（Cross Validation）：モデルを「たまたま当たった」で終わらせない

前回は、**特徴量エンジニアリング**を学びました。

* 特徴量（Feature）の考え方
* `PolynomialFeatures`
* Pipelineとの組み合わせ

今回は、機械学習実験で必須となる

> **Cross-validation（交差検証）**

を学びます。

ここから先は、「モデルを作る」だけでなく、

> **その評価をどれだけ信頼できるか**

という実験設計の考え方に入ります。

---

# 🎯 今日のゴール

授業終了時には、

* なぜ1回だけの train/test 分割では不十分なのか説明できる
* K-Fold Cross Validation の仕組みを理解できる
* `cross_val_score()` を使える
* モデル比較をより信頼できる形で行える

---

# 📖 講義（約20分）

## train/test分割の問題

これまで毎回、

```python
train_test_split()
```

を使っていました。

例えば

```
100件

↓

80件 train

20件 test
```

です。

しかし、

もし運悪く

```
簡単なデータだけ

↓

test
```

に入ったら？

Accuracyは高く見えてしまいます。

逆に、

難しいデータばかり入ると、

本当は良いモデルなのに低評価になります。

つまり、

**たった1回の分割だけでは偶然の影響を受けます。**

---

## Cross Validationとは？

例えば5分割します。

```
ABCDE
```

1回目

```
Train

BCDE

Test

A
```

---

2回目

```
Train

ACDE

Test

B
```

---

3回目

```
Train

ABDE

Test

C
```

……

これを全部繰り返します。

最後に

```
Accuracy

↓

平均
```

を取ります。

これが

**K-Fold Cross Validation**

です。

---

# 💻 実習1：データ準備

```python
import numpy as np

X = np.array([
    [1],
    [2],
    [3],
    [4],
    [5],
    [6],
    [7],
    [8],
    [9],
    [10]
])

y = np.array([
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    1
])
```

---

# 💻 実習2：モデル作成

```python
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
```

まだ

```
fit()
```

しません。

---

# 💻 実習3：cross_val_score

読み込み。

```python
from sklearn.model_selection import cross_val_score
```

実行。

```python
scores = cross_val_score(
    model,
    X,
    y,
    cv=5
)

print(scores)
```

例えば

```
[1.0
0.5
1.0
1.0
0.5]
```

のようになります。

これは

各FoldのAccuracyです。

---

# 💻 実習4：平均Accuracy

```python
print(
    scores.mean()
)
```

例えば

```
0.8
```

なら

平均80%

です。

---

## 標準偏差も見る

```python
print(
    scores.std()
)
```

これが小さいほど

安定しています。

---

# 📖 なぜ平均だけではダメ？

例えば

モデルA

```
95

95

94

96

95
```

平均95%。

---

モデルB

```
60

100

55

100

60
```

平均も83%くらいですが、

結果が非常に不安定です。

だから

```
平均

+

標準偏差
```

を確認します。

---

# 💻 実習5：Pipelineと組み合わせる

ここで前回のPipeline。

```python
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

pipe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression()
    )
])
```

これを

そのまま

```python
scores = cross_val_score(
    pipe,
    X,
    y,
    cv=5
)
```

できます。

Pipelineなので、

各Foldごとに

```
fit scaler

↓

transform

↓

fit model
```

が正しく行われます。

---

# 💻 実習6：cvを変える

例えば

```python
cv=3
```

や

```python
cv=10
```

でも動きます。

一般には

```
5

または

10
```

がよく使われます。

---

# 💻 実習7：回帰でも使える

線形回帰なら

```python
from sklearn.linear_model import LinearRegression
```

そして

```python
scores = cross_val_score(
    LinearRegression(),
    X,
    y,
    cv=5,
    scoring="r2"
)
```

回帰ではAccuracyではなく、

評価指標を

```
R²

neg_mean_squared_error

neg_mean_absolute_error
```

などに変更できます。

---

# 📖 Cross Validationは何をしている？

重要なのは

```
モデルを5個作っている

×

```

ではありません。

正確には

```
同じモデルを

5回学習

5回評価

↓

平均
```

です。

---

# ✍️ 演習

今日のデータ。

```python
X = np.array([
    [1],
    [2],
    [3],
    [4],
    [5],
    [6],
    [7],
    [8],
    [9],
    [10]
])

y = np.array([
    0,
    0,
    0,
    0,
    1,
    1,
    1,
    1,
    1,
    1
])
```

---

## 問1

LogisticRegressionを作ってください。

---

## 問2

5-Fold Cross Validation

を実行してください。

---

## 問3

Accuracyの平均を表示してください。

---

## 問4

Accuracyの標準偏差を表示してください。

---

## 問5

Pipelineを作り、

Cross Validationしてください。

---

## 問6（ボス戦👾）

次の関数を書いてください。

```python
def evaluate_cv(model, X, y):

    scores = ...

    print(...)

    print(...)
```

表示する内容。

* 各FoldのAccuracy
* 平均Accuracy
* 標準偏差

---

# 🌿 今日のまとめ

今日は、「評価の信頼性」を高めるための交差検証を学びました。

機械学習の開発フローは、ここまでで次のようにつながります。

```text
NumPy
      ↓
前処理
      ↓
特徴量作成
      ↓
Pipeline
      ↓
モデル学習
      ↓
評価
      ↓
Cross Validation ← ★今日
```

重要なのは、

> **「一度うまくいった」より、「何度やっても安定して良い結果が出る」**

ことです。

交差検証は、その安定性を確認するための基本的な方法であり、論文や実務でも広く使われています。

---

# 🔜 第19回予告

次回は、**ハイパーパラメータ探索**に進みます。

テーマは

* ハイパーパラメータとは何か
* Grid Search の考え方
* `GridSearchCV`
* 交差検証と組み合わせたモデル選択

です。

ここからは、「モデルを作る」だけでなく、**最適な設定をデータに基づいて選ぶ**という、実務の機械学習開発に一歩近づいていきます。